# 机器翻译作业总结

## 1. 本次作业的目标

本次作业的目标不是训练一个效果很好的机器翻译模型，而是理解 Seq2Seq 的完整工作流程：

```text
中文句子 → Encoder → 语义隐藏状态 → Decoder → 英文句子
```

作业的思路如下：

1. 获取中英文平行语料，对其分词和编码；
2. 分别构建中文词表和英文词表；
3. 构建 Dataset、DataLoader，完成动态 Padding 和 train/val 划分；
4. 构建 Encoder-Decoder 模型；
5. 理解训练阶段的教师强制；
6. 理解推理阶段的自回归和贪心策略；
7. 观察基础 Seq2Seq 的局限性，引出下一节课的 Attention 机制。

## 2. 数据预处理 data_process.py

流程如下：

```text
读取中英文平行语料
→ 中文先转换为简体，再用 jieba 分词
→ 英文使用 NLTK 的 TreebankWordTokenizer 分词
→ 处理 <SOS>、<EOS>、<PAD> 和 <UNK>
→ 分别构建中英文独立词表
→ 完成 encode 与 decode
```

四个特殊 Token 的作用：

| Token | 作用 |
| --- | --- |
| `<PAD>` | 将一个 Batch 中不同长度的句子补成相同长度 |
| `<SOS>` | 通知 Decoder 开始生成句子 |
| `<EOS>` | 表示一句话结束 |
| `<UNK>` | 表示词表中不存在的词 |

需要先划分 train/val，再只使用训练集构建中英文词表。验证集不参与词表统计，避免验证集信息提前泄露；验证集中的未登录词统一映射为 `<UNK>`。

当前中文训练词表大小为 11,265，英文训练词表大小为 7,453。两个词表长度不同是正常现象，因为中英文分词方式和词汇分布不同。

## 3. 构建数据集 dataset.py

流程如下：

```text
划分 train/val
→ 过滤超长句子
→ 提前完成分词和编码
→ 构造 Encoder/Decoder 序列
→ 在 collate_fn 中完成动态 Padding
```

Dataset 在初始化时一次性完成分词和编码，`__getitem__()` 只读取已经保存的 Tensor，避免每个 epoch 都重新运行 `jieba + zhconv`。

原始语料共有 26,388 对句子，划分为 23,750 条训练索引和 2,638 条验证索引；过滤 4 条超长句子后，实际进入 Dataset 的是 23,747 条训练样本和 2,637 条验证样本。

Encoder 输入为：

```text
中文内容 + <EOS>
```

Decoder 使用目标序列错开一位实现教师强制。假设完整目标为：

```text
<SOS> I love you <EOS>
```

则：

```text
decoder_input：<SOS> I    love you
target：          I  love you  <EOS>
```

训练时 Decoder 每一步看到正确的前一个英文词，并学习预测下一个词。动态 Padding 只把每个 Batch 补到当前 Batch 的最大长度，`src_lengths` 必须在补 PAD 之前记录。

## 4. 构建模型 model.py

整体流程如下：

```text
中文编号序列
→ Embedding + Dropout
→ Encoder RNN 得到最终 hidden
→ Decoder 以该 hidden 作为初始状态
→ 英文编号序列经过 Embedding + Dropout
→ Decoder RNN
→ Linear
→ 英文词表上的 logits
```

主要 Tensor 形状：

```text
src：            [batch_size, src_length]
src embedding：  [batch_size, src_length, embedding_dim]
encoder hidden： [num_layers, batch_size, hidden_size]
decoder_input：  [batch_size, target_length]
logits：         [batch_size, target_length, en_vocab_size]
```

计算交叉熵前，需要展开 Batch 和时间维：

```text
logits：[batch_size × target_length, en_vocab_size]
target：[batch_size × target_length]
```

- Encoder 使用 `pack_padded_sequence` 跳过句尾的 PAD，使最终 hidden 对应真实的 `<EOS>` 位置，而不是 PAD；
- 当前只有一层 RNN，`nn.RNN` 自带的 dropout 只作用于多层之间的连接，因此显式对词向量使用 `nn.Dropout(0.2)`；
- Seq2Seq 提供两种方式：训练使用 `forward`，通过教师强制一次算出整句所有位置的 logits；推理使用 `generate`，从 `<SOS>` 开始逐词生成，遇到 `<EOS>` 或达到最大长度时停止。

## 5. 训练 train.py

流程如下：

```text
加载固定的 train/val 划分和训练词表
→ DataLoader 动态 Padding
→ 教师强制前向传播
→ 计算交叉熵损失
→ 反向传播 + 梯度裁剪
→ Adam 更新参数
→ 验证集评估
→ 根据 val_loss 调整学习率
→ 保存最佳模型
```

- 交叉熵使用 `ignore_index=<PAD>`，因为 PAD 只是补齐符号，不参与损失计算；
- 梯度裁剪阈值为 1.0，用于减轻普通 RNN 的梯度爆炸；
- `ReduceLROnPlateau(patience=2)` 允许验证损失连续两轮不改善，第三轮仍未改善时将学习率减半；
- 只在 `val_loss` 创造新低时保存 `best_seq2seq.pt`；
- checkpoint 同时保存 `embedding_dim`、`hidden_size`、`num_layers` 和 `dropout`，推理时才能恢复出结构一致的模型。

损失公式为：

$$
L=-\frac{1}{N}\sum_t\log p(y_t\mid y_{<t},h)
$$

其中 $h$ 是 Encoder 的最终 hidden，$y_{<t}$ 是位置 $t$ 之前正确的英文词。

## 6. 推理 translate.py

流程如下：

```text
加载中英文词表
→ 读取 checkpoint
→ 重建 Encoder/Decoder 并 load_state_dict
→ model.eval() 关闭 Dropout
→ 中文编码：内容 + <EOS>
→ 从 <SOS> 开始自回归贪心生成
→ 遇到 <EOS> 停止
→ 英文词表 decode
→ TreebankWordDetokenizer 还原缩写和标点
```

训练与推理的核心区别：

| 阶段 | Decoder 上一步的输入 | 特点 |
| --- | --- | --- |
| 训练 | 正确的前一个英文词 | 教师强制，可以一次计算整句 logits |
| 推理 | 模型上一步预测的英文词 | 自回归，必须逐词生成 |

贪心策略表示每一步都直接选择 logits 最大的词。只要前面一步预测错误，后续 Decoder 就会继续基于错误结果生成。

## 7. 实验结果与局限性

两次实验对比：

- 无正则版本（20 轮）：`val_loss=3.2319@6`，之后单调恶化到 3.7162，而 train loss 仍在下降，属于典型过拟合；
- Dropout + 学习率调度版本：验证损失长期稳定在 3.25～3.30，最佳为 `3.2554@16`。后期没有快速恶化，但最佳值没有明显超过基础版本；
- 因此，Dropout 和学习率调度缓解了过拟合、稳定了训练曲线，但翻译效果的瓶颈不只来自过拟合，也来自基础 Seq2Seq 的结构限制。

需要注意：验证损失是在教师强制条件下计算的，只衡量“给定正确英文前文时预测下一个词”的能力；实际推理使用模型自己的历史预测。因此，`val_loss` 下降并不保证自回归翻译结果一定正确，最终仍然需要观察真实翻译样例。

推理现象：

- 贪心输出容易出现“开头看起来合理，后面逐渐跑偏”，有时连短句也会翻错；
- 模型对标点敏感：`我不想挂科` 翻错，而 `我不想挂科。` 翻对。训练集中存在带句号的完全一致样本，说明模型记住了样本，但泛化不足；
- 长句中的“非常适合”和“逛公园”不在训练词表中，会被映射成同一个 `<UNK>`，导致具体语义丢失。

局限的原因：

1. 固定长度语义瓶颈：整句中文必须压缩进最后一个 256 维 hidden，句子越长，前面的信息越容易丢失；
2. 教师强制与推理不一致（exposure bias）：前面一步预测错误，错误可能不断累积；
3. Decoder 容易学成一个弱英文语言模型，生成语法看似合理但与中文无关的句子；
4. `<UNK>` 无法区分不同的未登录词；语料中还存在一句多译和重复样本，增加了监督的不确定性。

## 8. 引出下一节课的 Attention 机制

基础 Seq2Seq 的 Decoder 只能拿到 Encoder 最后一个 hidden；Attention 会保留 Encoder 所有时间步的输出。Decoder 每生成一个词，都对中文各位置计算权重并进行加权求和，动态决定“当前应该看哪里”：

```text
生成 weather  → 重点关注“天气”
生成 suitable → 重点关注“适合”
生成 park     → 重点关注“公园”
```

这样就不必再让一个固定长度的向量保存整句话的信息。基础 Seq2Seq 解决的是“如何把一个序列转换成另一个序列”，Attention 接下来要解决的是“生成每个词时应该重点查看输入的哪个位置”。